In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_features, ModelTypes
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/Documents/phd/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_model = 'dv2'
model = get_model(selected_model, '../../trained_models', device=DEVICE, conf_path='../../dinov3')
n_dims = 768 if '_b' in selected_model else 384
# n_dims = 256

In [3]:
ds_folder = '../paper_figures/data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

replace_with_random_noise: bool = False

features = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, device=DEVICE, channel_last=True)
    features.append(feats)

In [4]:
preview_folder = '../paper_figures/data/linear_probe/'
preview_img_names = ['black_square_518.png', 'bulldog_518.png', 'edinburgh.png', '000.png']
preview_feats = []
preview_imgs: list[Image.Image] = []

for fname in preview_img_names:
    img_path = f'{preview_folder}/{fname}'
    img = Image.open(img_path).convert('RGB')
    img = img.resize((518, 518))
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, device=DEVICE, channel_last=True)
    preview_imgs.append(img)
    preview_feats.append(feats)


In [5]:
ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {r: [] for r in ramps}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for ramp in ramps:
    for i in range(n_imgs):
        feats = features[i]
        result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        ramps_to_results[ramp].append(result)

In [6]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [7]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)

In [8]:
# %%capture

plt.style.use("thesis.mplstyle")

plt.rcParams['lines.linewidth'] = 0.5
W, H = 7, 2.3 * 3.25

n_rows, n_cols = 9, 4
# TITLE_PAD = 20
# FS = 26

# Row 4 is a slim spacer between the top probe block and bottom preview block.
row_heights = [1, 1, 1, 1, 0.6, 1, 1, 1, 1]
col_widths = [1.0, 1, 1, 1.0]

fig = plt.figure(figsize=(W, H))
gs = GridSpec(
    n_rows,
    n_cols,
    figure=fig,
    height_ratios=row_heights,
    width_ratios=col_widths,
    hspace=0.3,
    wspace=0.15,
)

colors: dict[RampTypes, str] = {
    'lr': '#5762D5',
    'ud': '#6370C0',
    'diag': '#6E7DAB',
    'radial': '#575366',
}

color_map: dict[ModelTypes, dict[RampTypes, str]] = {
    'dv2': {
        'lr': '#5762D5',
        'ud': '#6370C0',
        'diag': '#6E7DAB',
        'radial': '#575366',
    },
    'vit_b': {
        'lr': "#E64444",
        'ud': "#DA5F5F",
        'diag': '#AB6060',
        'radial': '#665656',
    }
}

ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
}

FIG_A_START_ROW = 0
FIG_B_START_ROW = 5

# ---------- Top block: linear probe results (rows 0-3, cols 0-3) ----------
top_left_ramp_ax = None
for row, ramp in enumerate(ramps):
    gs_row = FIG_A_START_ROW + row
    h, w = 34, 34
    ramp_arr = get_ramp(ramp, h, w)

    ramp_ax = fig.add_subplot(gs[gs_row, 0])
    ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

    mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
    add_red_square_overlay(ramp_ax, mask, 1, 1)

    ramp_ax.set_xticks([])
    ramp_ax.set_yticks([])
    ramp_ax.set_ylabel(ramp_to_title[ramp])

    if row == 0:
        ramp_ax.set_title('Target ramp')
        top_left_ramp_ax = ramp_ax

worst_channels = []
for row, ramp in enumerate(ramps):
    gs_row = FIG_A_START_ROW + row
    score_ax = fig.add_subplot(gs[gs_row, 1:3])
    mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[ramp])

    print(f"{ramp}: {np.argsort(-mean_channel_scores)[:3]}")
    worst_channels.extend(list(np.argsort(-mean_channel_scores)[:2]))

    score_ax.hlines(0, 0, n_dims, 'red', '--')
    score_ax.plot(mean_channel_scores, color=color_map[selected_model][ramp])
    score_ax.fill_between(
        np.arange(n_dims),
        mean_channel_scores - std_channel_scores,
        mean_channel_scores + std_channel_scores,
        color=color_map[selected_model][ramp],
        alpha=0.3,
    )
    score_ax.set_ylim(-0.1, 1)
    score_ax.set_xlim(-10, n_dims + 10)
    score_ax.tick_params(axis='both')
    score_ax.grid(alpha=0.2)

    mean_pred_ax = fig.add_subplot(gs[gs_row, 3])
    mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

    if row == 0:
        score_ax.set_title('Per-channel ' + r'$R^2$' + ' scores')
        mean_pred_ax.set_title('Mean prediction')
    elif row == len(ramps) - 1:
        score_ax.set_xlabel('Channel')

    mean_pred_ax.set_ylabel(f'$R^{2}$: {mean_score:.2f}')
    mean_pred_ax.set_xticks([])
    mean_pred_ax.set_yticks([])

# ---------- Bottom block: channel previews (rows 5-8, cols 0-3) ----------
bad_channel_map = {
    'dv2': [47, 117, 359],
    'dvt': [55, 113, 188],
    'dv2_b': [39, 354, 480],
    'vit_b': [18, 390, 599],
    'dv': [293, 61, 89],
    'dv_b': [432, 266, 99],
    'dv3': [149, 123, 74],
    'clip_b': [69, 526, 554],
    'eva02': [180, 192, 180],
    'sam_b': [50, 27, 210],
    'deit': [173, 99, 302],
    'alibi_dv2_cb': [78, 73, 228],
}
bad_channels = bad_channel_map.get(selected_model, [47, 117, 359])

top_preview_ax = None
for i, img in enumerate(preview_imgs):
    gs_row = FIG_B_START_ROW + i

    img_ax = fig.add_subplot(gs[gs_row, 0])
    img_ax.imshow(img)
    img_ax.set_xticks([])
    img_ax.set_yticks([])
    if i == 0:
        top_preview_ax = img_ax
        img_ax.set_title('Input image')

    feats = preview_feats[i]
    for j, ch in enumerate(bad_channels):
        ch_ax = fig.add_subplot(gs[gs_row, j + 1])
        selected_ch = feats[:, :, ch]
        ch_ax.imshow(selected_ch, cmap='viridis')
        ch_ax.set_xticks([])
        ch_ax.set_yticks([])

        if i == 0:
            ch_ax.set_title(f'Ch. {ch}')

# labels = ['(a)', '(b)']
# axes = [top_left_ramp_ax, top_preview_ax]
# for i in range(2):
#     ax = axes[i]
#     text_str = "$\\textbf{" + labels[i] + "}$"
#     ax.text(
#         -0.55,
#         1.15,
#         text_str,
#         transform=ax.transAxes,
#         fontweight='bold',
#         color='black',
#     )

# plt.savefig('saved/02_vertical.png', dpi=300, bbox_inches='tight')
plt.savefig(f"out/linear_probe_{selected_model}.pdf", dpi=300, bbox_inches='tight')
plt.close()

lr: [117 359  44]
ud: [359 117  30]
diag: [117 345 208]
radial: [ 47 113  89]
